In [0]:
# Preparación: se ejecuta al iniciar o reconectar la sesión
from pyspark.sql import functions as F

# Las 9 tablas de Bronce, cargadas en un diccionario
tablas = ["customers", "orders", "order_items", "order_payments",
          "order_reviews", "products", "sellers", "geolocation",
          "category_translation"]
dfs = {t: spark.table(f"olist_proyecto.bronce.{t}") for t in tablas}

print("Tablas cargadas:", list(dfs.keys()))

In [0]:
# Lista de las 9 tablas de la capa Bronce
tablas = ["customers", "orders", "order_items", "order_payments",
          "order_reviews", "products", "sellers", "geolocation",
          "category_translation"]

# Cargamos cada tabla en un diccionario: dfs["orders"] devuelve la tabla orders
dfs = {t: spark.table(f"olist_proyecto.bronce.{t}") for t in tablas}

# Resumen rápido: filas y columnas de cada tabla
for nombre, df in dfs.items():
    print(f"{nombre}: {df.count()} filas, {len(df.columns)} columnas")

In [0]:
from pyspark.sql import functions as F

# Contamos, por tabla, las filas donde _rescued_data tiene algún contenido
for nombre, df in dfs.items():
    n = df.filter(F.col("_rescued_data").isNotNull()).count()
    print(f"{nombre}: {n} filas con _rescued_data")

In [0]:
# Cuenta, por columna, los valores nulos o vacíos de una tabla
def nulos_por_columna(df):
    cols = [c for c in df.columns if c != "_rescued_data"]
    return df.select([
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in cols
    ])

# Primero probamos con la tabla orders
display(nulos_por_columna(dfs["orders"]))

In [0]:
# Pedidos sin fecha de entrega al cliente, agrupados por estado
sin_entrega = F.col("order_delivered_customer_date").isNull() | \
              (F.trim(F.col("order_delivered_customer_date")) == "")

display(
    dfs["orders"]
    .filter(sin_entrega)
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
# Los 8 pedidos "delivered" sin fecha de entrega al cliente
display(
    dfs["orders"]
    .filter(sin_entrega & (F.col("order_status") == "delivered"))
    .drop("_rescued_data")
)

In [0]:
# Por tabla, muestra solo las columnas que tienen nulos o vacíos
for nombre, df in dfs.items():
    fila = nulos_por_columna(df).first().asDict()
    con_nulos = {c: n for c, n in fila.items() if n > 0}
    print(f"{nombre}: {con_nulos if con_nulos else 'sin nulos'}")

In [0]:
# Condición reutilizable: valor nulo o vacío
def vacio(c):
    return F.col(c).isNull() | (F.trim(F.col(c)) == "")

productos = dfs["products"]
sin_categoria = productos.filter(vacio("product_category_name"))

# ¿Son las mismas filas las que también carecen de nombre, descripción y fotos?
print("Sin categoría:", sin_categoria.count())
print("Sin categoría ni nombre, descripción y fotos:",
      sin_categoria.filter(vacio("product_name_lenght")
                           & vacio("product_description_lenght")
                           & vacio("product_photos_qty")).count())

In [0]:
items = dfs["order_items"]

# Líneas de venta cuyo producto es uno de los 610 sin ficha (cruce por product_id)
vendidos = items.join(sin_categoria.select("product_id"), on="product_id", how="inner")

print("Productos sin ficha:", sin_categoria.count())
print("De ellos, con al menos una venta:", vendidos.select("product_id").distinct().count())
print("Líneas de venta de esos productos:", vendidos.count())
print("Total de líneas de venta:", items.count())

In [0]:
# Tabla --> columna que debería identificar a cada fila de forma única
llaves = {"customers": "customer_id",
          "orders": "order_id",
          "products": "product_id",
          "sellers": "seller_id",
          "category_translation": "product_category_name"}

for tabla, llave in llaves.items():
    df = dfs[tabla]
    total = df.count()
    unicos = df.select(llave).distinct().count()
    print(f"{tabla}.{llave}: {total} filas, {unicos} únicos, {total - unicos} repetidos")

In [0]:
# Tabla --> columnas que juntas deberían identificar cada fila
llaves_compuestas = {"order_items": ["order_id", "order_item_id"],
                     "order_payments": ["order_id", "payment_sequential"],
                     "order_reviews": ["review_id"]}

for tabla, cols in llaves_compuestas.items():
    df = dfs[tabla]
    total = df.count()
    unicos = df.select(cols).distinct().count()   # combinaciones distintas
    print(f"{tabla} {cols}: {total} filas, {unicos} únicos, {total - unicos} repetidos")

In [0]:
reviews = dfs["order_reviews"].drop("_rescued_data")

# 1) Filas totalmente idénticas: si las repetidas fueran copias exactas, aquí bajaría el total
print("Filas totales:", reviews.count())
print("Filas distintas (todas las columnas):", reviews.distinct().count())

# 2) Cuántos review_id distintos se repiten
repetidos = reviews.groupBy("review_id").count().filter("count > 1")
print("review_id que se repiten:", repetidos.count())

# 3) Ejemplos: las filas de esos review_id, juntas para compararlas
display(
    reviews.join(repetidos.select("review_id"), on="review_id")
    .select("review_id", "order_id", "review_score", "review_creation_date")
    .orderBy("review_id")
    .limit(10)
)

In [0]:
reviews = dfs["order_reviews"].drop("_rescued_data")
total = reviews.count()

# ¿La combinación review_id + order_id es única?
combinaciones = reviews.select("review_id", "order_id").distinct().count()
print(f"review_id + order_id: {combinaciones} únicos de {total} filas")

# ¿Hay pedidos con más de una reseña?
pedidos = reviews.select("order_id").distinct().count()
print(f"order_id distintos: {pedidos}")

In [0]:
reviews = dfs["order_reviews"].drop("_rescued_data")

# Pedidos con más de una reseña (n_resenas = cuántas tiene cada uno)
multi = (reviews.groupBy("order_id").count()
         .withColumnRenamed("count", "n_resenas")
         .filter("n_resenas > 1"))

print("Pedidos con más de una reseña:", multi.count())

# Distribución: cuántos pedidos tienen 2, 3, ... reseñas
display(multi.groupBy("n_resenas").count().orderBy("n_resenas"))

# Ejemplos: las reseñas de esos pedidos, juntas para compararlas
display(
    reviews.join(multi.select("order_id"), on="order_id")
    .select("order_id", "review_id", "review_score",
            "review_creation_date", "review_answer_timestamp")
    .orderBy("order_id")
    .limit(10)
)

In [0]:
geo = dfs["geolocation"].drop("_rescued_data")

# Filas totales vs filas distintas (considerando todas las columnas)
print("Filas totales:", geo.count())
print("Filas distintas:", geo.distinct().count())

# Cuántos códigos postales distintos hay
print("Prefijos de código postal distintos:",
      geo.select("geolocation_zip_code_prefix").distinct().count())

In [0]:
# Trabajamos sobre filas distintas y convertimos las coordenadas a número
geo = dfs["geolocation"].drop("_rescued_data").distinct()
coords = geo.select(
    F.expr("try_cast(geolocation_lat AS double)").alias("lat"),
    F.expr("try_cast(geolocation_lng AS double)").alias("lng"))

# Mínimos y máximos, y cuántas coordenadas no se pudieron convertir
display(coords.select(
    F.min("lat").alias("lat_min"), F.max("lat").alias("lat_max"),
    F.min("lng").alias("lng_min"), F.max("lng").alias("lng_max"),
    F.sum(F.when(F.col("lat").isNull() | F.col("lng").isNull(), 1)
           .otherwise(0)).alias("no_convertibles")))

# Rango aproximado de Brasil: latitud de -34 a 6, longitud de -74 a -34
fuera = coords.filter(~F.col("lat").between(-34, 6) | ~F.col("lng").between(-74, -34))
print("Puntos fuera de Brasil:", fuera.count())

In [0]:
# Geolocation sin copias exactas y con coordenadas numéricas
geo_num = (dfs["geolocation"].drop("_rescued_data").distinct()
           .withColumn("lat", F.expr("try_cast(geolocation_lat AS double)"))
           .withColumn("lng", F.expr("try_cast(geolocation_lng AS double)")))

# Condición: punto dentro del rectángulo aproximado de Brasil
dentro = F.col("lat").between(-34, 6) & F.col("lng").between(-74, -34)

# Los 33 puntos fuera, con su ubicación declarada
fuera = geo_num.filter(~dentro)
display(fuera.select("geolocation_zip_code_prefix", "geolocation_city",
                     "geolocation_state", "lat", "lng").orderBy("lat"))

# ¿Algún prefijo afectado se queda sin ningún punto válido?
prefijos_malos = fuera.select("geolocation_zip_code_prefix").distinct()
prefijos_buenos = geo_num.filter(dentro).select("geolocation_zip_code_prefix").distinct()
sin_validos = prefijos_malos.join(prefijos_buenos, on="geolocation_zip_code_prefix", how="left_anti")

print("Prefijos afectados:", prefijos_malos.count())
print("Prefijos que quedarían sin ningún punto válido:", sin_validos.count())

In [0]:
# Rectángulo corregido: el límite este pasa de -34 a -28 (incluye Fernando de Noronha)
dentro = F.col("lat").between(-34, 6) & F.col("lng").between(-74, -28)

fuera = geo_num.filter(~dentro)
print("Puntos fuera de Brasil:", fuera.count())

# Prefijos afectados y cuántos se quedarían sin ningún punto válido
prefijos_malos = fuera.select("geolocation_zip_code_prefix").distinct()
prefijos_buenos = geo_num.filter(dentro).select("geolocation_zip_code_prefix").distinct()
sin_validos = prefijos_malos.join(prefijos_buenos, on="geolocation_zip_code_prefix", how="left_anti")

print("Prefijos afectados:", prefijos_malos.count())
print("Prefijos sin ningún punto válido:", sin_validos.count())
display(sin_validos)

In [0]:
# Lista de los 4 prefijos sin punto válido (collect trae el resultado a Python)
prefijos = [fila[0] for fila in sin_validos.collect()]
print("Prefijos:", prefijos)

# Cuántos clientes y vendedores usan alguno de esos prefijos
for tabla, col in [("customers", "customer_zip_code_prefix"),
                   ("sellers", "seller_zip_code_prefix")]:
    n = dfs[tabla].filter(F.col(col).isin(prefijos)).count()
    print(f"{tabla}: {n} filas con esos prefijos (de {dfs[tabla].count()})")

In [0]:
# (tabla hija, columna hija, tabla padre, columna padre)
relaciones = [
    ("orders",         "customer_id",           "customers",            "customer_id"),
    ("order_items",    "order_id",              "orders",               "order_id"),
    ("order_items",    "product_id",            "products",             "product_id"),
    ("order_items",    "seller_id",             "sellers",              "seller_id"),
    ("order_payments", "order_id",              "orders",               "order_id"),
    ("order_reviews",  "order_id",              "orders",               "order_id"),
    ("products",       "product_category_name", "category_translation", "product_category_name"),
]

for hija, c_h, padre, c_p in relaciones:
    h = dfs[hija].filter(F.col(c_h).isNotNull())   # ignoramos nulos, ya revisados
    valores_padre = dfs[padre].select(F.col(c_p).alias(c_h)).distinct()
    # left_anti: filas de la hija cuyo valor NO existe en el padre
    huerfanos = h.join(valores_padre, on=c_h, how="left_anti")
    print(f"{hija}.{c_h} -> {padre}: {huerfanos.count()} huérfanas de {h.count()}")

In [0]:
# Productos con categoría informada (no nula)
prod = dfs["products"].filter(F.col("product_category_name").isNotNull())

# Categorías que existen en la tabla de traducción
traducidas = dfs["category_translation"].select("product_category_name").distinct()

# Productos cuya categoría NO está en la traducción, agrupados por categoría
sin_traduccion = prod.join(traducidas, on="product_category_name", how="left_anti")
display(sin_traduccion.groupBy("product_category_name").count()
        .orderBy(F.desc("count")))

In [0]:
# Convertimos precio y flete de texto a número
items = dfs["order_items"].select(
    F.expr("try_cast(price AS double)").alias("price"),
    F.expr("try_cast(freight_value AS double)").alias("freight_value"))

for c in ["price", "freight_value"]:
    print(f"--- {c} ---")
    display(items.select(
        F.count(F.when(F.col(c).isNull(), 1)).alias("no_convertibles"),   # texto que no es número
        F.count(F.when(F.col(c) <= 0, 1)).alias("cero_o_negativos"),
        F.min(c).alias("minimo"),
        F.expr(f"percentile_approx({c}, 0.5)").alias("mediana"),
        F.expr(f"percentile_approx({c}, 0.99)").alias("p99"),            # el 99% de los valores está por debajo
        F.max(c).alias("maximo")))

In [0]:
# Líneas de venta con precio y flete numéricos, más la categoría del producto
lineas = (dfs["order_items"]
          .withColumn("price", F.expr("try_cast(price AS double)"))
          .withColumn("freight_value", F.expr("try_cast(freight_value AS double)"))
          .join(dfs["products"].select("product_id", "product_category_name"),
                on="product_id", how="left"))

# Las 10 líneas más caras
display(lineas.select("order_id", "product_category_name", "price", "freight_value")
        .orderBy(F.desc("price")).limit(10))

In [0]:
# Tabla --> columnas de fecha (guardadas como texto en Bronce)
fechas = {
    "orders": ["order_purchase_timestamp", "order_approved_at",
               "order_delivered_carrier_date", "order_delivered_customer_date",
               "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}

for tabla, cols in fechas.items():
    for c in cols:
        # Solo valores no vacíos, convertidos a fecha (NULL si no se puede)
        df = dfs[tabla]
        no_vacias = df.filter(F.col(c).isNotNull() & (F.trim(F.col(c)) != ""))
        conv = no_vacias.select(F.expr(f"try_cast({c} AS timestamp)").alias("f"))
        fallan = conv.filter(F.col("f").isNull()).count()
        rango = conv.agg(F.min("f"), F.max("f")).first()
        print(f"{tabla}.{c}: {fallan} no convertibles | de {rango[0]} a {rango[1]}")

In [0]:
# Fecha límite de envío y fecha de compra del pedido, ambas convertidas a fecha
env = (dfs["order_items"]
       .withColumn("limite_envio", F.expr("try_cast(shipping_limit_date AS timestamp)"))
       .join(dfs["orders"].select(
                 "order_id",
                 F.expr("try_cast(order_purchase_timestamp AS timestamp)").alias("compra")),
             on="order_id", how="left"))

# Líneas con fecha límite posterior a 2018 (el dataset termina ese año)
raras = env.filter(F.col("limite_envio") > "2018-12-31")
print("Líneas con fecha límite de envío posterior a 2018:", raras.count())

# Ejemplos: compra vs. fecha límite
display(raras.select("order_id", "compra", "limite_envio")
        .orderBy(F.desc("limite_envio")).limit(10))

In [0]:
# Fechas de orders convertidas a timestamp (NULL si no se puede)
cols_fecha = ["order_purchase_timestamp", "order_approved_at",
              "order_delivered_carrier_date", "order_delivered_customer_date"]
o = dfs["orders"].select(
    "order_id", "order_status",
    *[F.expr(f"try_cast({c} AS timestamp)").alias(c) for c in cols_fecha])

# Cada regla describe una incoherencia: cuántos pedidos la cumplen
reglas = {
    "aprobado antes de comprar":
        F.col("order_approved_at") < F.col("order_purchase_timestamp"),
    "enviado a transportadora antes de comprar":
        F.col("order_delivered_carrier_date") < F.col("order_purchase_timestamp"),
    "entregado al cliente antes de comprar":
        F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp"),
    "entregado al cliente antes de enviarse":
        F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"),
}

for nombre, cond in reglas.items():
    print(f"{nombre}: {o.filter(cond).count()} pedidos")

In [0]:
# 1) Enviado a transportadora antes de comprar: días de adelanto
c1 = (o.filter(reglas["enviado a transportadora antes de comprar"])
       .withColumn("dias_antes", F.round(
           (F.unix_timestamp("order_purchase_timestamp")
            - F.unix_timestamp("order_delivered_carrier_date")) / 86400, 2)))
print("Enviado antes de comprar (días de adelanto):")
display(c1.agg(F.min("dias_antes").alias("minimo"),
               F.expr("percentile_approx(dias_antes, 0.5)").alias("mediana"),
               F.max("dias_antes").alias("maximo")))
display(c1.select("order_id", "order_status", "order_purchase_timestamp",
                  "order_delivered_carrier_date", "dias_antes")
        .orderBy(F.desc("dias_antes")).limit(5))

# 2) Entregado al cliente antes de enviarse: días de adelanto
c2 = (o.filter(reglas["entregado al cliente antes de enviarse"])
       .withColumn("dias_antes", F.round(
           (F.unix_timestamp("order_delivered_carrier_date")
            - F.unix_timestamp("order_delivered_customer_date")) / 86400, 2)))
print("Entregado antes de enviarse (días de adelanto):")
display(c2.agg(F.min("dias_antes").alias("minimo"),
               F.expr("percentile_approx(dias_antes, 0.5)").alias("mediana"),
               F.max("dias_antes").alias("maximo")))

In [0]:
# Columnas categóricas a revisar: (tabla, columna)
categoricas = [("orders", "order_status"),
               ("order_payments", "payment_type"),
               ("order_reviews", "review_score"),
               ("customers", "customer_state")]

# Para cada una, los valores distintos y cuántas filas tiene cada uno
for tabla, c in categoricas:
    print(f"--- {tabla}.{c} ---")
    display(dfs[tabla].groupBy(c).count().orderBy(F.desc("count")))

In [0]:
# Los 3 pagos con método no definido
nd = dfs["order_payments"].filter(F.col("payment_type") == "not_defined").drop("_rescued_data")

# Se agrega el estado del pedido para entender el contexto
display(nd.join(dfs["orders"].select("order_id", "order_status"),
                on="order_id", how="left"))

In [0]:
# Pedidos cancelados que, aun así, tienen fecha de entrega al cliente
cancelados_entregados = dfs["orders"].filter(
    (F.col("order_status") == "canceled")
    & F.col("order_delivered_customer_date").isNotNull()
    & (F.trim(F.col("order_delivered_customer_date")) != ""))

print("Cancelados con fecha de entrega:", cancelados_entregados.count())
display(cancelados_entregados.select(
    "order_id", "order_purchase_timestamp",
    "order_delivered_customer_date", "order_estimated_delivery_date"))

In [0]:
# Nombre completo de las categorías sin traducción
prod = dfs["products"].filter(F.col("product_category_name").isNotNull())
traducidas = dfs["category_translation"].select("product_category_name").distinct()
sin_traduccion = prod.join(traducidas, on="product_category_name", how="left_anti")

# Se imprime como texto para que no se corte
for fila in sin_traduccion.select("product_category_name").distinct().collect():
    print(fila[0])